# Earnings-Event Intelligence — Research Notebook

**SYNTHETIC DATA.** This notebook reads the artifacts produced by `earnings-ml run-all` and walks through the research findings. All numbers come from the deterministic `MockProvider` (see `docs/ADR.md` ADR-001). They validate the pipeline and methodology; they are NOT claims about US equity markets.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from earnings_ml.config import load_config
from earnings_ml.utils import read_parquet, read_json

cfg = load_config()
A = cfg.artifacts_dir

## 1. Event master and effective-date alignment

In [ ]:
em = read_parquet(A / 'data/event_master.parquet')
print(f'{len(em)} events ({(em.exclusion_reason.fillna("")=="").sum()} primary)')
print(em.timing_class.value_counts())
print('\nGolden eff-date accuracy:', read_json(A/'metrics/event_validation.json')['golden']['effective_date_accuracy'])

## 2. Point-in-time invariant check

In [ ]:
fv = read_json(A / 'metrics/feature_validation.json')
print(f"PIT violations: {fv['pit_violations']} (must be 0)")

## 3. Walk-forward pooled metrics (reaction_car)

In [ ]:
tr = read_json(A / 'metrics/train_reaction_car.json')
rows = []
for m, v in tr['pooled_metrics'].items():
    rows.append({'model': m, 'pearson': v.get('pearson'), 'ci_low': v.get('pearson_ci_low'),
                 'ci_high': v.get('pearson_ci_high'), 'top_minus_bottom': v.get('top_minus_bottom'),
                 'roc_auc': v.get('roc_auc'), 'n': v.get('n_total')})
df = pd.DataFrame(rows).sort_values('pearson', ascending=False)
df

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
ax.barh(df.model, df.pearson, xerr=[df.pearson-df.ci_low, df.ci_high-df.pearson])
ax.set_xlabel('Pearson IC (reaction_car, pooled across folds, with 95% bootstrap CI)')
ax.axvline(0, color='k', lw=0.5)
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

## 4. Ablations — segment amplification and feature-group importance

In [ ]:
abl = read_json(A / 'metrics/ablation_reaction_car.json')
best = {k: max((mv.get('pearson', -9) or -9 for mv in models.values()), default=None)
        for k, models in abl['ablations'].items()}
pd.Series(best).sort_values().plot.barh(figsize=(8,5))
plt.xlabel('Best-model Pearson IC by ablation'); plt.tight_layout(); plt.show()

## 5. Signal decay across walk-forward folds

In [ ]:
decay = abl['robustness'].get('decay_per_fold', {})
if decay:
    s = pd.Series(decay['pearson_by_fold'])
    s.plot(marker='o', figsize=(7,4))
    plt.ylabel(f"Pearson IC ({decay['model']})"); plt.title('Signal decay across folds')
    plt.axhline(0, color='k', lw=0.5); plt.tight_layout(); plt.show()

## 6. Backtest — cost sweep

In [ ]:
for horizon in ['reaction', 'drift']:
    bt = read_json(A / f'backtest/lightgbm_{horizon}.json')
    sweep = pd.DataFrame(bt['cost_sweep']).T
    print(f'\n=== lightgbm / {horizon} (primary={bt["primary_cost_bps"]}bps) ===')
    print(sweep[['n_long','n_short','long_only_net','long_short_net']])

## 7. Conclusion

The framework's pipeline, point-in-time controls, leakage tests, walk-forward harness, and economic backtest all function correctly on synthetic data. The planted decaying signal is recovered (Pearson ~0.55-0.65 depending on segment and year), the negative control confirms no alpha is manufactured from noise, and the cost sweep shows the signal survives conservative transaction costs only marginally. **No empirical claim about US equities is made**; re-run with paid PIT-correct data for empirical conclusions.